# 04 - Evaluation & Benchmarking

This notebook conducts qualitative and quantitative benchmarking:
- **Quantitative Metrics**: PSNR (Peak Signal-to-Noise Ratio) & SSIM (Structural Similarity Index)
- **Visual Comparison**: Bicubic Interpolation vs SRCNN vs Ground Truth
- **Region-of-Interest (ROI) Zoom**: Fine texture and edge inspection

In [ ]:
import sys
sys.path.append("../src")

import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

from underwater_sr.model import SRCNN
from underwater_sr.metrics import calculate_psnr, calculate_ssim
from underwater_sr.utils import pil_to_tensor, tensor_to_pil, get_device

device = get_device("auto")
print(f"Device: {device}")

### Load Model & Test Image

In [ ]:
scale_factor = 4
model = SRCNN(scale_factor=scale_factor).to(device)
model.eval()

hr_img = Image.open("../sample_images/underwater_sample.png").convert("RGB")
lr_img = hr_img.resize((hr_img.width // scale_factor, hr_img.height // scale_factor), Image.Resampling.BICUBIC)

lr_t = pil_to_tensor(lr_img).unsqueeze(0).to(device)
hr_t = pil_to_tensor(hr_img).unsqueeze(0).to(device)

# Bicubic baseline
bicubic_t = F.interpolate(lr_t, scale_factor=float(scale_factor), mode="bicubic", align_corners=False).clamp(0.0, 1.0)

# Model prediction
with torch.no_grad():
    sr_t = model(lr_t)

# Compute metrics
psnr_bicubic = calculate_psnr(bicubic_t, hr_t)
ssim_bicubic = calculate_ssim(bicubic_t, hr_t)
psnr_sr = calculate_psnr(sr_t, hr_t)
ssim_sr = calculate_ssim(sr_t, hr_t)

print(f"Bicubic Baseline: PSNR = {psnr_bicubic:.2f} dB | SSIM = {ssim_bicubic:.4f}")
print(f"SRCNN Model:      PSNR = {psnr_sr:.2f} dB | SSIM = {ssim_sr:.4f}")

### Visual Comparison

In [ ]:
bicubic_img = tensor_to_pil(bicubic_t.squeeze(0))
sr_img = tensor_to_pil(sr_t.squeeze(0))

fig, axs = plt.subplots(1, 4, figsize=(18, 5))
axs[0].imshow(lr_img)
axs[0].set_title(f"Low-Res Input ({lr_img.size[0]}x{lr_img.size[1]})")
axs[0].axis("off")

axs[1].imshow(bicubic_img)
axs[1].set_title(f"Bicubic ({psnr_bicubic:.2f} dB)")
axs[1].axis("off")

axs[2].imshow(sr_img)
axs[2].set_title(f"SRCNN ({psnr_sr:.2f} dB)")
axs[2].axis("off")

axs[3].imshow(hr_img)
axs[3].set_title(f"Ground Truth HR ({hr_img.size[0]}x{hr_img.size[1]})")
axs[3].axis("off")
plt.tight_layout()
plt.show()